Експеримент 1: Перевірка ідеї в Notebook
Тема: Каталог тренувальних вправ
Основні поля:
- name (str) — назва вправи, обов’язкове поле.
- difficulty (str) — рівень складності: початковий, середній або високий.
- equipment (str) — обладнання, яке потрібне для виконання вправи, наприклад гантелі, штанга або нічого.
Мета: Перевірити, наскільки добре регулярні вирази та LLM можуть визначати назву вправи, її складність і необхідне обладнання на основі тексту користувача.

In [1]:
import re
import json
import pandas as pd

# 9 контрольних повідомлень з очікуваними значеннями (Ground Truth)
test_cases = [
    {
        "id": 1,
        "text": "Додай присідання зі штангою на плечах, це важка вправа високого рівня, треба штанга",
        "expected": {"name": "Присідання зі штангою", "difficulty": "високий", "equipment": "штанга"}
    },
    {
        "id": 2,
        "text": "Звичайні віджимання від підлоги, підійде для початківців, нічого з інвентарю не треба",
        "expected": {"name": "Віджимання від підлоги", "difficulty": "початковий", "equipment": "без обладнання"}
    },
    {
        "id": 3,
        "text": "Підтягування широким хватом на турніку, середній рівень",
        "expected": {"name": "Підтягування широким хватом", "difficulty": "середній", "equipment": "турнік"}
    },
    {
        "id": 4,
        "text": "Жим гантелей лежачи для середнього рівня складності, потрібні гантелі",
        "expected": {"name": "Жим гантелей лежачи", "difficulty": "середній", "equipment": "гантелі"}
    },
    {
        "id": 5,
        "text": "Плака для пресу на 1 хвилину",
        "expected": {"name": "Планка", "difficulty": None, "equipment": "без обладнання"} # Немає рівня
    },
    {
        "id": 6,
        "text": "Якась вправа з гантелями для ніг для новачків",
        "expected": {"name": None, "difficulty": "початковий", "equipment": "гантелі"} # Немає назви
    },
    {
        "id": 7,
        "text": "Станова тяга, високий рівень, робиться зі штангою",
        "expected": {"name": "Станова тяга", "difficulty": "високий", "equipment": "штанга"}
    },
    {
        "id": 8,
        "text": "Тренчик на біцепс з тренажером, просунутий рівень",
        "expected": {"name": "Згинання на біцепс", "difficulty": "високий", "equipment": "тренажер"}
    },
    {
        "id": 9,
        "text": "Випадання вперед з вагою, для середнього рівня",
        "expected": {"name": "Випади", "difficulty": "середній", "equipment": None} # Нечітке обладнання
    }
]

In [2]:
def extract_by_rules(text: str) -> dict:
    t = text.lower()
    
    # Визначаємо складність
    diff = None
    if "початк" in t or "новач" in t or "легк" in t:
        diff = "початковий"
    elif "середн" in t:
        diff = "середній"
    elif "висок" in t or "важк" in t or "просунут" in t:
        diff = "високий"
        
    # Визначаємо обладнання
    equip = None
    if "штанга" in t or "штангою" in t:
        equip = "штанга"
    elif "гантел" in t:
        equip = "гантелі"
    elif "турнік" in t:
        equip = "турнік"
    elif "без обладнання" in t or "нічого" in t or "від підлоги" in t:
        equip = "без обладнання"
    elif "тренажер" in t:
        equip = "тренажер"

    # Назву вправи регулярками нормалізовано витягнути дуже важко
    name = None
    if "віджимання" in t:
        name = "Віджимання від підлоги"
    elif "станова тяга" in t:
        name = "Станова тяга"
    elif "підтягування" in t:
        name = "Підтягування"

    return {
        "name": name,
        "difficulty": diff,
        "equipment": equip
    }

In [3]:
def mock_llm_call(text: str, prompt_ver: str = "A") -> dict:
    """Імітація відповіді LLM для перевірки контракту без грошей/API"""
    t = text.lower()
    
    if "присідання зі штангою" in t:
        # Уявимо, що модель помилилася в обладнанні у версії А
        eq = "гиря" if prompt_ver == "A" else "штанга"
        return {"name": "Присідання зі штангою", "difficulty": "високий", "equipment": eq}
    if "віджимання" in t:
        return {"name": "Віджимання від підлоги", "difficulty": "початковий", "equipment": "без обладнання"}
    if "підтягування" in t:
        return {"name": "Підтягування широким хватом", "difficulty": "середній", "equipment": "турнік"}
    if "жим гантелей" in t:
        return {"name": "Жим гантелей лежачи", "difficulty": "середній", "equipment": "гантелі"}
    if "плака" in t: # Описка в тексті
        name = "Планка" if prompt_ver == "B" else "Плака"
        return {"name": name, "difficulty": None, "equipment": "без обладнання"}
    if "якась вправа" in t:
        return {"name": None, "difficulty": "початковий", "equipment": "гантелі"}
    if "станова тяга" in t:
        return {"name": "Станова тяга", "difficulty": "високий", "equipment": "штанга"}
    if "тренчик" in t:
        return {"name": "Згинання рук на біцепс", "difficulty": "високий", "equipment": "тренажер"}
    if "випадання" in t:
        return {"name": "Випади вперед", "difficulty": "середній", "equipment": "гайка"} # Помилка LLM

    return {"name": None, "difficulty": None, "equipment": None}


def validate_exercise(proposal: dict, original_text: str) -> tuple[bool, str]:
    """Перевіряє пропозицію LLM перед тим як дозволити збереження"""
    # 1. Перевірка наявності назви
    if not proposal.get("name"):
        return False, "Відсутня назва вправи (потрібне уточнення)"
        
    # 2. Перевірка допустимих значень для складності
    valid_diffs = ["початковий", "середній", "високий", None]
    if proposal.get("difficulty") not in valid_diffs:
        return False, f"Невідомий рівень складності: {proposal.get('difficulty')}"
        
    # 3. Перевірка галюцинацій в обладнанні
    eq = proposal.get("equipment")
    if eq and eq not in ["без обладнання", "гайка"]: 
        # Перевіряємо чи є згадка інвентарю в коді/тексті
        if eq == "гиря" and "штанга" in original_text.lower():
            return False, "Обладнання 'гиря' суперечить тексту ('штанга')"
            
    if eq == "гайка":
        return False, "Некоректна назва обладнання"

    return True, "OK"

In [4]:
results = []
for item in test_cases:
    r_res = extract_by_rules(item["text"])
    llm_res = mock_llm_call(item["text"], prompt_ver="A")
    is_ok, reason = validate_exercise(llm_res, item["text"])
    
    results.append({
        "ID": item["id"],
        "Вхід": item["text"],
        "Правила (Name)": r_res["name"],
        "LLM (Name)": llm_res["name"],
        "LLM Equip": llm_res["equipment"],
        "Валідація": "PASSED" if is_ok else f"REJECTED ({reason})"
    })

df = pd.DataFrame(results)
df

,ID,Вхід,Правила (Name),LLM (Name),LLM Equip,Валідація
0,1,"Додай присідання зі штангою на плечах, це важк...",NaN,Присідання зі штангою,гиря,REJECTED (Обладнання 'гиря' суперечить тексту ...
1,2,"Звичайні віджимання від підлоги, підійде для п...",Віджимання від підлоги,Віджимання від підлоги,без обладнання,PASSED
2,3,"Підтягування широким хватом на турніку, середн...",Підтягування,Підтягування широким хватом,турнік,PASSED
3,4,Жим гантелей лежачи для середнього рівня склад...,NaN,Жим гантелей лежачи,гантелі,PASSED
4,5,Плака для пресу на 1 хвилину,NaN,Плака,без обладнання,PASSED
5,6,Якась вправа з гантелями для ніг для новачків,NaN,NaN,гантелі,REJECTED (Відсутня назва вправи (потрібне уточ...
6,7,"Станова тяга, високий рівень, робиться зі штангою",Станова тяга,Станова тяга,штанга,PASSED
7,8,"Тренчик на біцепс з тренажером, просунутий рівень",NaN,Згинання рук на біцепс,тренажер,PASSED
8,9,"Випадання вперед з вагою, для середнього рівня",NaN,Випади вперед,гайка,REJECTED (Некоректна назва обладнання)


In [5]:
# Тестуємо різницю промптів на випадку з опискою ("Плака" -> "Планка")
case_typo = test_cases[4]["text"]

p_a = mock_llm_call(case_typo, prompt_ver="A")
p_b = mock_llm_call(case_typo, prompt_ver="B")

print("--- Результати експерименту з промптами ---")
print("Промпт A (простий):", p_a)
print("Промпт B (з інструкцією виправляти описки):", p_b)

--- Результати експерименту з промптами ---
Промпт A (простий): {'name': 'Плака', 'difficulty': None, 'equipment': 'без обладнання'}
Промпт B (з інструкцією виправляти описки): {'name': 'Планка', 'difficulty': None, 'equipment': 'без обладнання'}


Висновки досліду

1. Де працюють прості правила: Звичайні if/else та регулярні вирази допомагають знаходити ключові слова й визначати рівень складності та необхідний інвентар. Але іноді вони можуть неправильно зрозуміти контекст.

2. Де потрібна LLM: Вона допомагає правильно сформулювати назву вправи та краще зрозуміти текст, якщо в ньому є сленг або помилки.

3. Що має перевіряти сервіс: По-перше, назва вправи обов'язкова. Якщо її немає, сервіс має повернути помилку й не додавати запис до сховища. По-друге, рівень складності може бути тільки початковим, середнім або високим. По-третє, потрібно перевіряти, чи відповідає вказаний інвентар початковому тексту, щоб модель не додавала того, чого користувач не зазначав.